# Pemodelan LSTM untuk Prediksi Air Quality Index (AQI)

Notebook ini digunakan untuk membangun dan mengevaluasi model
Long Short-Term Memory (LSTM) dalam memprediksi AQI harian.

Eksperimen menggunakan perangkat Jl. Lapangan Rajawali,
dengan pembagian data training dan testing yang sama seperti
eksperimen SARIMA.

Evaluasi model menggunakan MAE, RMSE, dan MAPE.

# 1. Memeriksa Library

## 1.1 Memeriksa tensorflow

In [1]:
# Langkah 1: Memeriksa library dan environment

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)

try:
    import tensorflow as tf

    print("TensorFlow:", tf.__version__)
    print("GPU tersedia:", tf.config.list_physical_devices("GPU"))

except ImportError:
    print("TensorFlow belum terpasang pada environment ini.")

Pandas: 3.0.3
NumPy: 2.5.1
TensorFlow belum terpasang pada environment ini.


In [2]:
import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("Perangkat GPU:", tf.config.list_physical_devices("GPU"))

TensorFlow: 2.21.0
Perangkat GPU: []


# 2. Menyiapkan data LSTM

In [8]:
# Periode yang sama dengan eksperimen SARIMA
tanggal_mulai = "2025-11-03"
tanggal_akhir = "2026-02-18"

aqi_harian = aqi_harian.loc[
    tanggal_mulai:tanggal_akhir
]

print("Jumlah data:", len(aqi_harian))
print("Tanggal awal:", aqi_harian.index.min())
print("Tanggal akhir:", aqi_harian.index.max())
print("Jumlah nilai kosong:", aqi_harian.isna().sum())

Jumlah data: 108
Tanggal awal: 2025-11-03 00:00:00
Tanggal akhir: 2026-02-18 00:00:00
Jumlah nilai kosong: 0


# 3. Pembagian data latih dan data uji

- Data latih: 86 hari, dari 3 November 2025 sampai 27 Januari 2026.
- Data uji: 22 hari, dari 28 Januari sampai 18 Februari 2026.

In [5]:
# Membagi data secara kronologis
jumlah_train = 86

train = aqi_harian.iloc[:jumlah_train]
test = aqi_harian.iloc[jumlah_train:]

print("Jumlah data latih:", len(train))
print("Periode latih:", train.index.min(), "sampai", train.index.max())

print("\nJumlah data uji:", len(test))
print("Periode uji:", test.index.min(), "sampai", test.index.max())

Jumlah data latih: 86
Periode latih: 2025-11-03 00:00:00 sampai 2026-01-27 00:00:00

Jumlah data uji: 22
Periode uji: 2026-01-28 00:00:00 sampai 2026-02-18 00:00:00


# 4. Normalisasi data menggunakan minmaxscaller

- fit_transform() pada data latih mempelajari nilai minimum dan maksimum dari data latih, lalu mengubah skalanya.
- transform() pada data uji menggunakan parameter yang sama tanpa mempelajari informasi dari data uji.
- Nilai data uji bisa berada di luar rentang 0–1 apabila terdapat AQI yang lebih rendah atau lebih tinggi daripada nilai minimum atau maksimum data latih. Itu normal dan tidak perlu dipaksakan masuk ke rentang tersebut.

In [6]:
from sklearn.preprocessing import MinMaxScaler

# Membuat scaler
scaler = MinMaxScaler(feature_range=(0, 1))

# Scaler hanya belajar dari data training
train_scaled = scaler.fit_transform(
    train.to_numpy().reshape(-1, 1)
)

# Data testing menggunakan scaler yang sama
test_scaled = scaler.transform(
    test.to_numpy().reshape(-1, 1)
)

print("Bentuk data training:", train_scaled.shape)
print("Bentuk data testing:", test_scaled.shape)

print("\nRentang training:")
print("Minimum:", train_scaled.min())
print("Maksimum:", train_scaled.max())

print("\nRentang testing:")
print("Minimum:", test_scaled.min())
print("Maksimum:", test_scaled.max())

Bentuk data training: (86, 1)
Bentuk data testing: (22, 1)

Rentang training:
Minimum: 0.0
Maksimum: 0.9999999999999999

Rentang testing:
Minimum: -0.05987334626750976
Maksimum: 0.5110090357841698


# 5. Membentuk Sequence

- LSTM memerlukan data dalam bentuk urutan waktu.
- menggunakan 7 hari sebelum untuk memprediksi AQI pada hari berikutnya

In [9]:
def buat_sequence(data, look_back=7):
    X, y = [], []

    for i in range(len(data) - look_back):
        X.append(data[i:i + look_back, 0])
        y.append(data[i + look_back, 0])

    return np.array(X), np.array(y)


look_back = 7

# Sequence untuk data training
X_train, y_train = buat_sequence(
    train_scaled, look_back
)

# Data testing diawali dengan 7 hari terakhir training
data_test_gabungan = np.concatenate(
    [train_scaled[-look_back:], test_scaled],
    axis=0
)

X_test, y_test = buat_sequence(
    data_test_gabungan, look_back
)

print("X_train:", X_train.shape)
print("y_train:", y_train.shape)
print("X_test:", X_test.shape)
print("y_test:", y_test.shape)

X_train: (79, 7)
y_train: (79,)
X_test: (22, 7)
y_test: (22,)


- data training yang semula 86 menjadi 79, karena 7 hari pertama digunakan sebagai riwayat input.
- data testing tetap memiliki 22 sequence karena menggunakan 7 hari terakhir training untuk memprediksi seluruh periode testing

# 6. Reshape 3D

LSTM membutuhkan input dengan format:
- (jumlah sampel, jumlah timestep, jumlah fitur)

Pada penelitian ini:
- Jumlah sampel training: 79.
- Timestep: 7 hari.
- Fitur: 1, yaitu AQI.

In [10]:
# Mengubah input menjadi format 3 dimensi untuk LSTM
X_train_lstm = X_train.reshape(
    X_train.shape[0],
    X_train.shape[1],
    1
)

X_test_lstm = X_test.reshape(
    X_test.shape[0],
    X_test.shape[1],
    1
)

print("Bentuk X_train_lstm:", X_train_lstm.shape)
print("Bentuk X_test_lstm:", X_test_lstm.shape)

Bentuk X_train_lstm: (79, 7, 1)
Bentuk X_test_lstm: (22, 7, 1)


# 7. Membangun model LSTM

In [18]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Input, LSTM, Dense
from tensorflow.keras.optimizers import Adam

model_lstm = Sequential([
    Input(shape=(7, 1)),
    LSTM(32, activation="tanh"),
    Dense(1)
])

model_lstm.compile(
    optimizer=Adam(learning_rate=0.001),
    loss="mean_squared_error"
)

model_lstm.summary()

Model: "sequential_2"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ lstm_2 (LSTM)                   │ (None, 32)             │         4,352 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 1)              │            33 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 4,385 (17.13 KB)

 Trainable params: 4,385 (17.13 KB)

 Non-trainable params: 0 (0.00 B)

# 8. Melatih model

- epochs=100: batas maksimum, bukan keharusan untuk menjalankan seluruh epoch.
- batch_size=8: ukuran batch kecil untuk menghemat sumber daya.
- validation_split=0.2: memisahkan sebagian data training untuk memantau generalisasi.
- shuffle=False: mempertahankan urutan sampel deret waktu.
- restore_best_weights=True: mengembalikan bobot pada epoch dengan nilai validasi terbaik.

In [19]:
from tensorflow.keras.callbacks import EarlyStopping

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

history = model_lstm.fit(
    X_train_lstm,
    y_train,
    epochs=100,
    batch_size=8,
    validation_split=0.2,
    shuffle=False,
    callbacks=[early_stopping],
    verbose=1
)

print("Jumlah epoch dijalankan:", len(history.history["loss"]))
print("Training loss terakhir:", history.history["loss"][-1])
print("Validation loss terakhir:", history.history["val_loss"][-1])

Epoch 1/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 2s 41ms/step - loss: 0.1110 - val_loss: 0.0218
Epoch 2/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0749 - val_loss: 0.0115
Epoch 3/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0607 - val_loss: 0.0091
Epoch 4/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0592 - val_loss: 0.0092
Epoch 5/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0576 - val_loss: 0.0092
Epoch 6/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 0.0564 - val_loss: 0.0093
Epoch 7/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0564 - val_loss: 0.0094
Epoch 8/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0561 - val_loss: 0.0093
Epoch 9/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 14ms/step - loss: 0.0556 - val_loss: 0.0094
Epoch 10/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0551 - val_loss: 0.0095
Epoch 11/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0546 - val_loss: 0.0096
Epoch 12/100
8/8 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - loss: 0.0542 - val_lo

## Hasil
- Epoch maksimum	100
- Epoch dijalankan	13
- Training loss terakhir	0,05655
- Validation loss terakhir	0,01221
- Validation loss terbaik	0,0108 (epoch ke-3)

# 9. Prediksi data testing

In [13]:
# Prediksi data testing dalam skala normalisasi
y_pred_scaled = model_lstm.predict(
    X_test_lstm,
    verbose=0
)

# Mengembalikan prediksi ke skala AQI asli
y_pred = scaler.inverse_transform(
    y_pred_scaled
).flatten()

# Mengembalikan nilai aktual ke skala AQI asli
y_test_actual = scaler.inverse_transform(
    y_test.reshape(-1, 1)
).flatten()

# Membuat tabel hasil prediksi
hasil_lstm = pd.DataFrame({
    "Tanggal": test.index,
    "AQI_Aktual": y_test_actual,
    "AQI_Prediksi": y_pred
})

print(hasil_lstm.to_string(index=False))

   Tanggal  AQI_Aktual  AQI_Prediksi
2026-01-28   17.951087     21.541065
2026-01-29   31.074866     22.272762
2026-01-30   27.474359     23.193113
2026-01-31   10.000000     23.795719
2026-02-01    1.640000     23.450310
2026-02-02   10.356688     22.652313
2026-02-03    8.025478     20.972425
2026-02-04    4.783439     19.525921
2026-02-05   12.936306     18.737690
2026-02-06    7.068182     17.447950
2026-02-07   20.629834     16.310280
2026-02-08   21.613260     16.649052
2026-02-09   30.650273     17.635588
2026-02-10   13.596685     18.580061
2026-02-11   11.583333     19.241501
2026-02-12    5.347826     19.926346
2026-02-13    3.668539     19.760115
2026-02-14    5.917160     19.691448
2026-02-15   14.964497     18.761236
2026-02-16   10.331034     18.053482
2026-02-17   32.234940     16.812969
2026-02-18   37.314607     17.383631


- Pada 1 Februari, AQI aktual sebesar 1,64, sedangkan prediksi sekitar 23,45.
- Pada 9 Februari, AQI aktual sebesar 30,65, sedangkan prediksi sekitar 17,64.
- Pada 18 Februari, AQI aktual sebesar 37,31, sedangkan prediksi sekitar 17,38.

# 10. Menghitung MAE, RMSE, MAPE

In [14]:
from sklearn.metrics import mean_absolute_error, mean_squared_error
import numpy as np

# Menghitung metrik evaluasi pada skala AQI asli
mae_lstm = mean_absolute_error(
    y_test_actual, y_pred
)

rmse_lstm = np.sqrt(
    mean_squared_error(y_test_actual, y_pred)
)

mape_lstm = np.mean(
    np.abs((y_test_actual - y_pred) / y_test_actual)
) * 100

evaluasi_lstm = pd.DataFrame({
    "Model": ["LSTM"],
    "MAE": [mae_lstm],
    "RMSE": [rmse_lstm],
    "MAPE (%)": [mape_lstm]
})

print(evaluasi_lstm.round(4).to_string(index=False))

Model     MAE    RMSE  MAPE (%)
 LSTM 10.6683 11.9064  165.7383


In [20]:
hasil_lstm.to_csv(
    "prediksi_lstm.csv",
    index=True,
    index_label="updated_at"
)

print("Prediksi LSTM berhasil disimpan.")
print(hasil_lstm.head())

Prediksi LSTM berhasil disimpan.
     Tanggal  AQI_Aktual  AQI_Prediksi
0 2026-01-28   17.951087     21.541065
1 2026-01-29   31.074866     22.272762
2 2026-01-30   27.474359     23.193113
3 2026-01-31   10.000000     23.795719
4 2026-02-01    1.640000     23.450310
